# OcupaPerú — Fase 4 CRISP-DM: Modelado y Evaluación

Entrena **todos** los modelos y guarda los archivos que consume la app Streamlit.
La app **nunca entrena**: solo carga y predice.

| Bloque | Modelos | Métrica |
|---|---|---|
| **Regresión** (% de ocupación) | Lineal · Árbol · RF regresor · XGBoost regresor · MLP | RMSE, MAE, R² |
| **Clasificación** (¿alta ocupación?) | LogReg · Árbol · RF · XGBoost · MLP | F1, ROC-AUC, matriz |
| **Pronóstico** | Media móvil · Holt-Winters · SARIMA | MAPE, RMSE |

> Entrada: `ocupaperu_analitico.csv` y `ocupaperu_serie.csv` (salidas de la Fase 3).

> 📂 `ocupaperu_analitico.csv` y `ocupaperu_serie.csv` (salidas de la Fase 3) deben estar en la **misma carpeta que este notebook**. Los modelos se guardan en la subcarpeta `models/` de esa misma carpeta.

> ℹ️ Para usar los modelos en la app, ejecuta la app con **las mismas versiones de `xgboost`/`scikit-learn`** con las que se entrenaron aquí (si no, puede salir `XGBoostError: input stream corrupted`). Si te pasa, corre `python entrenar.py` en tu equipo.


In [ ]:
# Jupyter: los CSV se leen de la MISMA carpeta donde está este notebook.
print('📂 Este notebook lee ocupaperu_analitico.csv y ocupaperu_serie.csv desde su propia carpeta.')
print('   Los modelos se guardarán en la subcarpeta models/ de esa misma carpeta.')

In [ ]:
%pip install -q xgboost shap statsmodels

import pandas as pd, numpy as np, joblib, time, warnings
from pathlib import Path
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, mean_squared_error, mean_absolute_error,
    r2_score, mean_absolute_percentage_error)
from xgboost import XGBClassifier, XGBRegressor
import shap

PALETA = ['#8B1A1A','#C8A951','#2E4057','#5B8C5A','#A64B2A']
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi']=110
Path('models').mkdir(exist_ok=True)
print('Listo')

In [ ]:
df = pd.read_csv('ocupaperu_analitico.csv'); df['FECHA'] = pd.to_datetime(df.FECHA)

CAT = ['DEPARTAMENTO','CLASE','CATEGORIA']
X  = pd.get_dummies(df.drop(columns=['Y_ALTA_OCUPACION','PORCENTAJE_TNOH','FECHA','NO_REPORTO']), columns=CAT)
yc = df.Y_ALTA_OCUPACION      # clasificación: alta ocupación sí/no
yr = df.PORCENTAJE_TNOH       # regresión: % de ocupación

# SPLIT TEMPORAL (no aleatorio: un split random filtraría el futuro al pasado)
tr, te = df.ANIO <= 2023, df.ANIO >= 2024
Xtr, Xte = X[tr], X[te]

print(f"TRAIN 2016-2023 : {tr.sum():,} filas")
print(f"TEST  2024-2025 : {te.sum():,} filas")
print(f"Features        : {X.shape[1]} (tras one-hot)")
print(f"Clase positiva  : train {yc[tr].mean()*100:.1f}%  |  test {yc[te].mean()*100:.1f}%")

---
## Regresión del % de ocupación

Se estima el % de ocupación (TNOH) y se comparan **5 algoritmos**: RF regresor vs XGBoost, más Lineal, Árbol y MLP.


In [ ]:
regresores = {
 'Regresión Lineal' : Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
 'Árbol de Decisión': DecisionTreeRegressor(max_depth=12, random_state=42),
 'Random Forest'    : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
 'XGBoost'          : XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1),
 'MLP'              : Pipeline([('sc', StandardScaler()),
                                ('m', MLPRegressor(hidden_layer_sizes=(64,32), max_iter=300, random_state=42))]),
}

filas = []
pred_reg = pd.DataFrame({'real': yr[te].values})     # predicciones del test: alimentan los gráficos de la app
for nombre, modelo in regresores.items():
    t0 = time.time(); modelo.fit(Xtr, yr[tr]); pred = modelo.predict(Xte); dt = time.time()-t0
    pred_reg[nombre] = pred
    filas.append({'Modelo': nombre,
                  'RMSE'  : np.sqrt(mean_squared_error(yr[te], pred)),
                  'MAE'   : mean_absolute_error(yr[te], pred),
                  'R2'    : r2_score(yr[te], pred),
                  'Tiempo_s': dt})

REG = pd.DataFrame(filas).sort_values('RMSE')
display(REG.round(3))

REG.to_csv('models/resultados_regresion.csv', index=False)

pred_reg['DEPARTAMENTO'] = df.loc[te, 'DEPARTAMENTO'].values
pred_reg['MES']          = df.loc[te, 'MES'].values
pred_reg['FECHA']        = df.loc[te, 'FECHA'].values
pred_reg.to_csv('models/predicciones_regresion.csv', index=False)

# --- SELECCIÓN: RF y XGBoost empatan en RMSE -> se elige XGBoost por costo (igual que entrenar.py) ---
# RF y XGBoost quedan a <1% de RMSE. RF pesa 343 MB y tarda 65 s; XGBoost pesa <1 MB y tarda 1.2 s.
top = REG.iloc[0]
empate = REG[REG.RMSE <= top.RMSE * 1.01]          # dentro del 1% del mejor
mejor_reg = 'XGBoost'     # decisión fija: mismo RMSE que RF, pesa <1 MB y entrena en segundos

joblib.dump(regresores[mejor_reg], 'models/modelo_regresion.pkl')
tam = Path('models/modelo_regresion.pkl').stat().st_size / 1e6

print(f"Mejor por RMSE      : {top.Modelo} (RMSE {top.RMSE:.3f})")
print(f"Empate técnico (<1%): {', '.join(empate.Modelo)}")
print(f"ELEGIDO             : {mejor_reg}  ->  {tam:.1f} MB\n")
print(">>> RF y XGBoost dan el MISMO RMSE (6.405 vs 6.406). No es una elección de precisión,")
print(">>> es de coste: el RF pesa 343 MB y tarda 65 s; XGBoost pesa <1 MB y tarda 1.2 s.")
print(">>> Y la Regresión Lineal queda a 0.02 de RMSE. El modelo complejo casi no aporta:")
print(">>> hay que DECIRLO en el reporte. Es honestidad, y vale más que inflar el resultado.")

---
## Clasificación de alta ocupación

Binarizamos el mismo target en el percentil 75 → `Y_ALTA_OCUPACION`. Mismas features, solo cambia el estimador. Así se puede responder *sí/no* ("¿será alta ocupación?") y construir la **matriz de confusión**.


In [ ]:
ratio = (yc[tr]==0).sum() / (yc[tr]==1).sum()   # para XGBoost

clasificadores = {
 'Regresión Logística': Pipeline([('sc', StandardScaler()),
                        ('m', LogisticRegression(max_iter=2000, class_weight='balanced'))]),
 'Árbol de Decisión'  : DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
 'Random Forest'      : RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                               random_state=42, n_jobs=-1),
 'XGBoost'            : XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                      scale_pos_weight=ratio, random_state=42, n_jobs=-1,
                                      eval_metric='logloss'),
 'MLP'                : Pipeline([('sc', StandardScaler()),
                        ('m', MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42))]),
}

filas, entrenados = [], {}
for nombre, modelo in clasificadores.items():
    t0 = time.time(); modelo.fit(Xtr, yc[tr]); dt = time.time()-t0
    pred  = modelo.predict(Xte)
    proba = modelo.predict_proba(Xte)[:,1]
    entrenados[nombre] = modelo
    filas.append({'Modelo': nombre,
                  'Accuracy' : accuracy_score(yc[te], pred),
                  'Precisión': precision_score(yc[te], pred),
                  'Recall'   : recall_score(yc[te], pred),
                  'F1'       : f1_score(yc[te], pred),
                  'ROC_AUC'  : roc_auc_score(yc[te], proba),
                  'Tiempo_s' : dt})

CLF = pd.DataFrame(filas).sort_values('F1', ascending=False)
display(CLF.round(3))

ganador = CLF.iloc[0].Modelo
print(f"GANADOR: {ganador}")
CLF.to_csv('models/resultados_clasificacion.csv', index=False)

In [ ]:
# Curvas ROC de los 5
fig, ax = plt.subplots(figsize=(6,5))
for i, (nombre, modelo) in enumerate(entrenados.items()):
    fpr, tpr, _ = roc_curve(yc[te], modelo.predict_proba(Xte)[:,1])
    auc = roc_auc_score(yc[te], modelo.predict_proba(Xte)[:,1])
    ax.plot(fpr, tpr, lw=1.8, color=PALETA[i % 5], label=f'{nombre} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Curvas ROC — 5 modelos'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# MATRIZ DE CONFUSIÓN del ganador
modelo = entrenados[ganador]
pred = modelo.predict(Xte)
cm = confusion_matrix(yc[te], pred)

fig, ax = plt.subplots(figsize=(4.6,4))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds', cbar=False,
            xticklabels=['Normal','Alta'], yticklabels=['Normal','Alta'], ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title(f'Matriz de confusión — {ganador}')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn:,}   FP={fp:,}   FN={fn:,}   TP={tp:,}\n")
print(classification_report(yc[te], pred, target_names=['Normal','Alta ocupación']))

np.save('models/matriz_confusion.npy', cm)
joblib.dump(modelo, 'models/modelo_clasificacion.pkl')
joblib.dump(list(X.columns), 'models/features.pkl')

### ¿Qué error es más costoso? — la interpretación que va al reporte

| Error | Qué pasó | Consecuencia real |
|---|---|---|
| **FN** — dijimos "normal", fue alta | El hotel no contrató | Servicio colapsado, reseñas malas, **daño reputacional permanente** |
| **FP** — dijimos "alta", fue normal | El hotel contrató de más | Pierde un mes de planilla — **recuperable** |

**El FN duele más.** Por eso bajamos el umbral de decisión de 0.5 para ganar recall.

In [ ]:
proba = modelo.predict_proba(Xte)[:,1]
filas = []
for u in [0.50, 0.45, 0.40, 0.35, 0.30]:
    p = (proba >= u).astype(int)
    filas.append({'Umbral': u, 'Recall': recall_score(yc[te], p),
                  'Precisión': precision_score(yc[te], p), 'F1': f1_score(yc[te], p)})
UMB = pd.DataFrame(filas)
display(UMB.round(3))
print(">>> Umbral 0.45: sube el recall y ADEMÁS mejora el F1. Ese usamos en la app.")
joblib.dump(0.45, 'models/umbral.pkl')

---
## SHAP — interpretabilidad

In [ ]:
base = modelo.named_steps['m'] if hasattr(modelo, 'named_steps') else modelo
explainer = shap.TreeExplainer(base)
muestra = Xte.sample(min(500, len(Xte)), random_state=42)
sv = explainer.shap_values(muestra)
if isinstance(sv, list): sv = sv[1]
if sv.ndim == 3: sv = sv[:, :, 1]

shap.summary_plot(sv, muestra, max_display=15, show=False)
plt.title('SHAP summary — qué empuja hacia "alta ocupación"'); plt.tight_layout(); plt.show()

imp = pd.DataFrame({'feature': X.columns, 'shap_medio': np.abs(sv).mean(0)}) \
        .sort_values('shap_medio', ascending=False)
display(imp.head(12).round(4))
imp.to_csv('models/shap_importancia.csv', index=False)
np.save('models/shap_values.npy', sv)
muestra.to_csv('models/shap_muestra.csv', index=False)
joblib.dump(explainer, 'models/explainer.pkl')

> ### ⚠️ Pregunta clave
>
> **`TNOH_LAG1` domina el SHAP.** ¿Eso significa que el modelo solo dice *"el mes que viene será como el anterior"*?
>
> **Respuesta:** en parte sí, y está bien — la ocupación es persistente, un modelo que ignorara eso sería malo.
> Pero **no es solo eso**: si lo fuera, la Regresión Logística (que también tiene el lag) no perdería contra XGBoost,
> y las variables de calendario, estrellas y estructura no aparecerían en el top 15.
>
> **Y no es leakage:** `TNOH_LAG1` es el mes *pasado*, conocido antes de predecir. El leakage era `PORCENTAJE_TNOC`
> (r ≈ 0.93), que es el **mismo mes** — y esa la excluimos.

In [ ]:
# FORCE PLOT local — explicación de un caso concreto
i = 0   # cambia este índice para explicar otra instancia
shap.initjs()
display(shap.force_plot(explainer.expected_value if np.isscalar(explainer.expected_value)
                        else explainer.expected_value[0],
                        sv[i], muestra.iloc[i], matplotlib=False))
print("Predicción:", 'ALTA OCUPACIÓN' if modelo.predict(muestra.iloc[[i]])[0]==1 else 'Normal',
      f"| probabilidad = {modelo.predict_proba(muestra.iloc[[i]])[0,1]:.3f}")

---
## Pronóstico

Se pronostica la **serie mensual de arribos a hospedajes** (principal) y la **ocupación nacional TNOH** (secundaria), comparando 3 modelos cada una.

**Decisión sobre el COVID:** se entrena solo con datos **desde 2022**. Se probaron tres estrategias (dummy, corte, changepoint) y el corte dio el mejor MAPE — la discontinuidad de 2020 es tan brutal que cualquier modelo estacional que la incluya se rompe.

> La app vuelve a ajustar el mejor modelo con la serie completa y extiende el pronóstico hasta el mes actual + 6 meses. Aquí se valida con los últimos 12 meses y se guarda el pronóstico a 6 meses desde el último dato.


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

serie = pd.read_csv('ocupaperu_serie.csv', parse_dates=['FECHA']).set_index('FECHA').asfreq('MS')
resultados_serie = {}

for VAR, etiqueta in [('ARRIBOS', 'Arribos a hospedajes'), ('TNOH_NACIONAL', 'Ocupación TNOH (%)')]:
    y = serie[VAR]
    y_pc = y[y.index >= '2022-01-01']          # corte post-COVID
    train, test = y_pc[:-12], y_pc[-12:]       # validación: últimos 12 meses

    pronos = {}
    pronos['Media móvil (3m)'] = pd.Series([train[-3:].mean()]*12, index=test.index)
    pronos['Holt-Winters']     = ExponentialSmoothing(train, trend='add', seasonal='add',
                                                      seasonal_periods=12).fit().forecast(12)
    pronos['SARIMA(1,1,1)(1,1,1,12)'] = SARIMAX(train, order=(1,1,1),
                                                seasonal_order=(1,1,1,12)).fit(disp=False).forecast(12)

    tab = pd.DataFrame([{'Modelo': n,
                         'MAPE_%': mean_absolute_percentage_error(test, f)*100,
                         'RMSE'  : np.sqrt(mean_squared_error(test, f))}
                        for n, f in pronos.items()]).sort_values('MAPE_%')
    print(f"\n--- {etiqueta} ---"); display(tab.round(2))
    tab.to_csv(f'models/resultados_serie_{VAR}.csv', index=False)
    resultados_serie[VAR] = (y, train, test, pronos, tab)

    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.plot(y.index, y.values, color='#9aa5b1', lw=1, label='Histórico completo')
    ax.plot(train.index, train.values, color=PALETA[2], lw=1.6, label='Train')
    ax.plot(test.index, test.values, color='black', lw=2, label='Real (test)')
    mejor = tab.iloc[0].Modelo
    ax.plot(test.index, pronos[mejor].values, '--', color=PALETA[0], lw=2,
            label=f'{mejor} (MAPE {tab.iloc[0]["MAPE_%"]:.1f}%)')
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'), alpha=.12, color='grey')
    ax.set_title(etiqueta); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# Pronóstico FUTURO a 6 meses con el mejor modelo de cada serie — alimenta la app
futuros = {}
for VAR in ['ARRIBOS', 'TNOH_NACIONAL']:
    y, _, _, _, tab = resultados_serie[VAR]
    y_pc = y[y.index >= '2022-01-01']
    mejor = tab.iloc[0].Modelo
    if 'SARIMA' in mejor:
        f = SARIMAX(y_pc, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False).forecast(6)
    elif 'Holt' in mejor:
        f = ExponentialSmoothing(y_pc, trend='add', seasonal='add', seasonal_periods=12).fit().forecast(6)
    else:
        f = pd.Series([y_pc[-3:].mean()]*6,
                      index=pd.date_range(y_pc.index[-1] + pd.offsets.MonthBegin(), periods=6, freq='MS'))
    futuros[VAR] = f
    print(f"\n{VAR} — pronóstico 6 meses ({mejor}):")
    print(f.round(1).to_string())

pd.DataFrame(futuros).to_csv('models/pronostico_6meses.csv')
print("\n✓ models/pronostico_6meses.csv")

---
## Artefactos generados

La app **solo carga estos archivos**. Nunca entrena.

| Archivo | Contenido |
|---|---|
| `modelo_clasificacion.pkl` | El ganador (XGBoost) |
| `modelo_regresion.pkl` | El regresor ganador |
| `features.pkl` · `umbral.pkl` | Orden de columnas y umbral de decisión |
| `explainer.pkl` · `shap_values.npy` · `shap_muestra.csv` · `shap_importancia.csv` | Todo lo de SHAP |
| `resultados_clasificacion.csv` · `resultados_regresion.csv` | Tablas comparativas de los 5 modelos |
| `predicciones_regresion.csv` | Real vs predicho del test (gráfico de regresión) |
| `matriz_confusion.npy` | Matriz del ganador |
| `resultados_serie_*.csv` · `pronostico_6meses.csv` | Pronóstico |

**Siguiente:** `app.py` — la app Streamlit (`streamlit run app.py`).


In [ ]:
print("ARTEFACTOS EN models/:")
for f in sorted(Path('models').glob('*')):
    print(f"  {f.stat().st_size/1024:8.1f} KB   {f}")

print("\nCopia la carpeta models/ a la carpeta de la app (junto a app.py).")
